<a href="https://colab.research.google.com/github/sdkfjuecsec/ProteinMPNN/blob/main/colab_notebooks/quickdemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ProteinMPNN
This notebook is intended as a quick demo, more features to come!

In [1]:
#@title Clone github repo
# Import standard-library modules used for filesystem access, timing, and path management
import json, time, os, sys, glob

# Only clone the repository if it has not already been downloaded
if not os.path.isdir("ProteinMPNN"):
  # Quietly clone the ProteinMPNN repository from GitHub
  os.system("git clone -q https://github.com/sdkfjuecsec/ProteinMPNN.git")
# Add the cloned repository root to the Python import path so its modules are discoverable
sys.path.append('/content/ProteinMPNN')

In [2]:
#@title Setup Model
# Import plotting and filesystem/utility libraries
import matplotlib.pyplot as plt
import shutil
import warnings
# Import numerical computing and PyTorch deep-learning libraries
import numpy as np
import torch
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.data.dataset import random_split, Subset
import copy
import torch.nn as nn
import torch.nn.functional as F
import random
import os.path
# Import typing helper for annotating heterogeneous checkpoint dictionaries
from typing import Any
# Import ProteinMPNN utilities: losses, graph helpers, featurization, and model class
from protein_mpnn_utils import loss_nll, loss_smoothed, gather_edges, gather_nodes, gather_nodes_t, cat_neighbors_nodes, _scores, _S_to_seq, tied_featurize, parse_PDB
from protein_mpnn_utils import StructureDataset, StructureDatasetPDB, ProteinMPNN, TiedFeatures

# Select the GPU when available, otherwise fall back to CPU
device: torch.device = torch.device("cuda:0" if (torch.cuda.is_available()) else "cpu")
#v_48_010=version with 48 edges 0.10A noise
model_name: str = "v_48_020" #@param ["v_48_002", "v_48_010", "v_48_020", "v_48_030"]


# Standard deviation of Gaussian noise added to backbone atom coordinates during inference
backbone_noise: float = 0.00               # Standard deviation of Gaussian noise to add to backbone atoms

# Path to the directory holding the pretrained (vanilla) model weights
path_to_model_weights: str = '/content/ProteinMPNN/vanilla_model_weights'
# Hidden feature dimension shared across encoder and decoder layers
hidden_dim: int = 128
# Number of transformer layers in both the encoder and decoder
num_layers: int = 3
# Local alias of the weights folder used to build the checkpoint path
model_folder_path: str = path_to_model_weights
# Guarantee the folder path ends with a trailing slash for clean concatenation
if model_folder_path[-1] != '/':
    model_folder_path = model_folder_path + '/'
# Assemble the full checkpoint file path for the selected model variant
checkpoint_path: str = model_folder_path + f'{model_name}.pt'

# Load the checkpoint dictionary from disk, mapping tensors to the chosen device
checkpoint: dict[str, Any] = torch.load(checkpoint_path, map_location=device)
# Report the number of k-nearest-neighbour edges stored in the checkpoint
print('Number of edges:', checkpoint['num_edges'])
# Retrieve the noise level used during training from the checkpoint metadata
noise_level_print: float = checkpoint['noise_level']
# Display the training noise level in Angstroms
print(f'Training noise level: {noise_level_print}A')
# Instantiate the ProteinMPNN model with the configured hyperparameters and checkpoint edge count
model: ProteinMPNN = ProteinMPNN(num_letters=21, node_features=hidden_dim, edge_features=hidden_dim, hidden_dim=hidden_dim, num_encoder_layers=num_layers, num_decoder_layers=num_layers, augment_eps=backbone_noise, k_neighbors=checkpoint['num_edges'])
# Move the model parameters to the selected device (GPU or CPU)
model.to(device)
# Load the pretrained weights from the checkpoint into the model
model.load_state_dict(checkpoint['model_state_dict'])
# Switch the model to evaluation mode (disables dropout and gradient tracking)
model.eval()
# Confirm that the model weights were loaded successfully
print("Model loaded")

Number of edges: 48
Training noise level: 0.2A
Model loaded


In [ ]:
#@title Helper functions
# Import typing helper for annotating heterogeneous PDB dictionaries
from typing import Any

def make_tied_positions_for_homomers(pdb_dict_list: list[dict[str, Any]]) -> dict[str, list[dict[str, list[int]]]]:
    """Build tied-positions mapping so homomer chains stay sequence-identical.

    For every parsed PDB structure, each residue position across all chains is
    tied to the same position index on every other chain, forcing the sampler
    to produce identical sequences on symmetric (homomer) chains.

    Args:
        pdb_dict_list: List of parsed PDB dictionaries (one per structure) as
            produced by ``parse_PDB``. Each dict carries ``seq_chain_<X>``
            entries and a ``name`` field.

    Returns:
        Mapping from structure name to a list of tied-position dicts. Each
        tied-position dict maps a chain ID (e.g. ``"A"``) to a single-element
        list holding the 1-based residue index that should be tied.

        Example for a homotrimer named '1O91' with chains A, B, C, each
        containing 3 residues:

            {
                '1O91': [
                    {'A': [1], 'B': [1], 'C': [1]},
                    {'A': [2], 'B': [2], 'C': [2]},
                    {'A': [3], 'B': [3], 'C': [3]},
                ]
            }
    """
    # Accumulator mapping each structure name to its list of tied positions
    tied_positions_dict: dict[str, list[dict[str, list[int]]]] = {}
    # Iterate over each parsed PDB structure dictionary
    for pdb_dict in pdb_dict_list:
        # Extract and sort single-character chain IDs from the seq_chain_* keys (A, B, C, ...)
        all_chain_id_list: list[str] = sorted([key[-1:] for key in list(pdb_dict) if key[:9]=='seq_chain']) #A, B, C, ...
        # List of per-position tied mappings for the current structure
        tied_positions_list: list[dict[str, list[int]]] = []
        # Length of the first chain (assumes equal-length homomer chains)
        chain_length: int = len(pdb_dict[f"seq_chain_{all_chain_id_list[0]}"])
        # Build a tied mapping for every 1-based residue position
        for i in range(1, chain_length+1):
            # Temporary dict collecting the tied index per chain at this position
            temp_dict: dict[str, list[int]] = {}
            # Assign the current position index to every chain (value must be a list)
            for chain in all_chain_id_list:
                temp_dict[chain] = [i] #needs to be a list
            # Store this position's tied mapping in the structure's list
            tied_positions_list.append(temp_dict)
        # Associate the completed tied-positions list with the structure's name
        tied_positions_dict[pdb_dict['name']] = tied_positions_list
    # Return the complete tied-positions dictionary
    return tied_positions_dict

Examples:

1) pdb: 6MRR, homomer: False, designed_chain: A

2) pdb: 1O91, homomer: True, designed_chain: A B C, for correct symmetric tying lenghts of homomer chains should be the same

In [ ]:
# Import regular-expression utilities for splitting chain-id strings
import re
# Import the Colab file-upload widget for loading user-supplied PDB files
from google.colab import files
# Import NumPy for array-based amino-acid mask construction
import numpy as np
# Import typing helper for annotating heterogeneous PDB dictionaries
from typing import Any
# Import the canonical amino-acid alphabet from the central constants module
from constants import PROTEIN_ALPHABET

#########################
def get_pdb(pdb_code: str = "") -> str:
  """Resolve a PDB structure to a local ``.pdb`` file path.

  When ``pdb_code`` is empty the user is prompted to upload a PDB file via
  the Colab widget; otherwise the structure is downloaded from the RCSB
  Protein Data Bank using the supplied 4-character code.

  Args:
      pdb_code: A 4-character RCSB PDB code (e.g. ``"1O91"``). Pass an empty
          string (or ``None``) to trigger an interactive file upload.

  Returns:
      The local file path to the resolved ``.pdb`` file - ``"tmp.pdb"`` for
      uploads or ``"<pdb_code>.pdb"`` for downloads.
  """
  # If no PDB code was supplied, fall back to an interactive upload
  if pdb_code is None or pdb_code == "":
    # Launch the Colab upload widget and capture the returned file mapping
    upload_dict = files.upload()
    # Grab the raw bytes of the first (and only) uploaded file
    pdb_string = upload_dict[list(upload_dict.keys())[0]]
    # Persist the uploaded bytes to a temporary local PDB file
    with open("tmp.pdb","wb") as out: out.write(pdb_string)
    # Return the path to the freshly written temporary file
    return "tmp.pdb"
  else:
    # Quietly download the requested PDB file from the RCSB server
    os.system(f"wget -qnc https://files.rcsb.org/view/{pdb_code}.pdb")
    # Return the path to the downloaded PDB file
    return f"{pdb_code}.pdb"

#@markdown ### Input Options
# RCSB PDB code of the target structure (leave blank to upload a file instead)
pdb: str = '1O91' #@param {type:"string"}
# Resolve the PDB code (or upload) into a local file path
pdb_path: str = get_pdb(pdb)
#@markdown - pdb code (leave blank to get an upload prompt)

# Whether the target is a homomer whose symmetric chains should be tied
homomer: bool = True #@param {type:"boolean"}
# Space-separated chain IDs to redesign (e.g. "A B C")
designed_chain: str = "A B C" #@param {type:"string"}
# Space-separated chain IDs to keep fixed during design (e.g. "D E")
fixed_chain: str = "" #@param {type:"string"}

# Parse the designed-chain string into a list, handling empty input
if designed_chain == "":
  # No chains to design yields an empty list
  designed_chain_id_list: list[str] = []
else:
  # Replace non-letter separators with commas and split into chain IDs
  designed_chain_id_list = re.sub("[^A-Za-z]+",",", designed_chain).split(",")

# Parse the fixed-chain string into a list, handling empty input
if fixed_chain == "":
  # No chains to fix yields an empty list
  fixed_chain_id_list: list[str] = []
else:
  # Replace non-letter separators with commas and split into chain IDs
  fixed_chain_id_list = re.sub("[^A-Za-z]+",",", fixed_chain).split(",")

# Build the de-duplicated, order-preserving list of all chains (designed + fixed)
chain_list: list[str] = list(dict.fromkeys(designed_chain_id_list + fixed_chain_id_list))

#@markdown - specified which chain(s) to design and which chain(s) to keep fixed.
#@markdown   Use comma:`A,B` to specifiy more than one chain

#chain = "A" #@param {type:"string"}
#pdb_path_chains = chain
##@markdown - Define which chain to redesign

#@markdown ### Design Options
# Number of sequence samples to generate per target backbone
num_seqs: int = 1 #@param ["1", "2", "4", "8", "16", "32", "64"] {type:"raw"}
# Alias of num_seqs used by the downstream sampling loop
num_seq_per_target: int = num_seqs

#@markdown - Sampling temperature for amino acids, T=0.0 means taking argmax, T>>1.0 means sample randomly.
# Sampling temperature(s) controlling amino-acid sampling randomness
sampling_temp: str = "0.1" #@param ["0.0001", "0.1", "0.15", "0.2", "0.25", "0.3", "0.5"]



# Flag: save per-position score (-log probability) to .npy files
save_score: int = 0                      # 0 for False, 1 for True; save score=-log_prob to npy files
# Flag: save MPNN predicted probabilities per position
save_probs: int = 0                      # 0 for False, 1 for True; save MPNN predicted probabilites per position
# Flag: only score the input backbone-sequence pairs (no sampling)
score_only: int = 0                      # 0 for False, 1 for True; score input backbone-sequence pairs
# Flag: output conditional probabilities p(s_i given the rest of the sequence and backbone)
conditional_probs_only: int = 0          # 0 for False, 1 for True; output conditional probabilities p(s_i given the rest of the sequence and backbone)
# Flag: output conditional probabilities p(s_i given backbone only)
conditional_probs_only_backbone: int = 0 # 0 for False, 1 for True; if true output conditional probabilities p(s_i given backbone)

# Number of sequences processed per GPU batch (reduce if out of memory)
batch_size: int = 1                      # Batch size; can set higher for titan, quadro GPUs, reduce this if running out of GPU memory
# Maximum allowed total residue count per target
max_length: int = 20000                  # Max total residue count

# Directory where output sequences will be written
out_folder: str = '.'                    # Path to a folder to output sequences, e.g. /home/out/
# Optional path to a folder of pre-parsed PDBs in jsonl format
jsonl_path: str = ''                     # Path to a folder with parsed pdb into jsonl
# Amino acids to exclude from generated sequences (e.g. 'AC' omits Ala and Cys)
omit_AAs: str = 'X'                      # Specify which amino acids should be omitted in the generated sequence, e.g. 'AC' would omit alanine and cystine.

# Weight blending PSSM bias with MPNN predictions in [0.0, 1.0]
pssm_multi: float = 0.0                    # A value between [0.0, 1.0], 0.0 means do not use pssm, 1.0 ignore MPNN predictions
# Threshold restricting which AAs are allowed per position when using PSSM
pssm_threshold: float = 0.0                # A value between -inf + inf to restric per position AAs
# Flag: use PSSM log-odds as the bias source
pssm_log_odds_flag: int = 0               # 0 for False, 1 for True
# Flag: apply per-residue PSSM bias during sampling
pssm_bias_flag: int = 0                   # 0 for False, 1 for True


##############################################################

# Output folder derived from the out_folder setting
folder_for_outputs: str = out_folder

# Number of full batches needed to produce the requested number of sequences
NUM_BATCHES: int = num_seq_per_target//batch_size
# Number of independent copies run per batch (mirrors batch_size)
BATCH_COPIES: int = batch_size
# List of float sampling temperatures parsed from the comma-separated string
temperatures: list[float] = [float(item) for item in sampling_temp.split()]
# String of amino acids to omit (kept as a raw string for membership tests)
omit_AAs_list: str = omit_AAs
# Canonical amino-acid alphabet imported from the central constants module
alphabet: str = PROTEIN_ALPHABET

# Boolean mask (float32) marking which alphabet positions are omitted
omit_AAs_np: np.ndarray = np.array([AA in omit_AAs_list for AA in alphabet]).astype(np.float32)

# Optional per-structure chain-id mapping (populated later); None means use defaults
chain_id_dict = None
# Optional per-structure fixed-position mapping; None means no fixed residues
fixed_positions_dict = None
# Optional per-structure PSSM mapping; None means no PSSM bias
pssm_dict = None
# Optional per-structure omit-AA mapping; None means no per-structure omits
omit_AA_dict = None
# Optional per-residue AA bias mapping; None means no bias
bias_AA_dict = None
# Optional per-structure tied-positions mapping; None means no tying
tied_positions_dict = None
# Optional per-residue bias mapping; None means no per-residue bias
bias_by_res_dict = None
# Zero-initialised per-alphabet AA bias vector used during sampling
bias_AAs_np: np.ndarray = np.zeros(len(alphabet))


###############################################################
# Parse the target PDB file into a list of per-structure dictionaries
pdb_dict_list: list[dict[str, Any]] = parse_PDB(pdb_path, input_chain_list=chain_list)
# Wrap the parsed structures in a StructureDatasetPDB for batched iteration
valid_pdb_dataset: StructureDatasetPDB = StructureDatasetPDB(pdb_dict_list, truncate=None, max_length=max_length)

# Build the chain-id dict mapping the structure name to (designed_chain_id_list, fixed_chain_id_list) lists
chain_id_dict = {}
# Map the first structure's name to its designed and fixed chain lists
chain_id_dict[pdb_dict_list[0]['name']]= (designed_chain_id_list, fixed_chain_id_list)

# Print the chain-id mapping for inspection
print(chain_id_dict)
# Report the length of each chain in the parsed structure
for chain in chain_list:
  # Length (residue count) of the current chain
  l: int = len(pdb_dict_list[0][f"seq_chain_{chain}"])
  # Print the chain ID and its residue count
  print(f"Length of chain {chain} is {l}")

# If the target is a homomer, build tied positions to enforce symmetry
if homomer:
  # Generate tied-positions dict constraining homomer chains to identical sequences
  tied_positions_dict = make_tied_positions_for_homomers(pdb_dict_list)
  print(f"Tied positions for homomer design: {tied_positions_dict}")
else:
  # No tying for non-homomer targets
  tied_positions_dict = None

{'1O91': (['A', 'B', 'C'], [])}
Length of chain C is 131
Length of chain A is 131
Length of chain B is 131


In [ ]:
#@title RUN
# Disable gradient tracking: inference-only sampling, saves memory and compute
with torch.no_grad():
  # Announce the start of sequence generation
  print('Generating sequences...')
  # Iterate over every protein structure in the dataset
  # structure_index: shape = scalar; pdb_dict: shape = dict[str, Any] (one parsed PDB entry: name, seq, coords)
  for structure_index, pdb_dict in enumerate(valid_pdb_dataset):
    # score_list: scores (-log probability) of every sequence sampled for this structure,
    # shape = list[float] (one scalar per sample)
    score_list: list[float] = []
    # all_probs_list: temperature-adjusted sampling probabilities collected for visualization,
    # shape = list[np.ndarray], each element shape = [B, L, 21]
    all_probs_list: list[np.ndarray] = []
    # all_log_probs_list: raw per-position log probabilities collected for visualization,
    # shape = list[np.ndarray], each element shape = [B, L, 21]
    all_log_probs_list: list[np.ndarray] = []
    # sampled_sequence_list: sampled sequences (amino-acid index tensors) collected for analysis,
    # shape = list[np.ndarray], each element shape = [B, L]
    sampled_sequence_list: list[np.ndarray] = []
    # batch_clones: the protein entry deep-copied once per batch copy,
    # shape = list[dict], length = B = BATCH_COPIES
    batch_clones: list[dict[str, Any]] = [copy.deepcopy(pdb_dict) for _ in range(BATCH_COPIES)]
    # Featurize the batch into padded tensors, with B = BATCH_COPIES and L = max_length.
    # Shape of every returned field, read as features.<field>:
    #   features.X: shape = [B, L, 4, 3] (N/CA/C/O atoms; [B, L, 3] in ca_only mode)
    #   features.S: shape = [B, L] (0-based indices into PROTEIN_ALPHABET)
    #   features.mask: shape = [B, L] (1.0 for a valid residue, 0.0 for padding or missing coordinates)
    #   features.sequence_lengths: shape = [B] (int32 total residue count per entry)
    #   features.chain_M: shape = [B, L] (chain-level design mask)
    #   features.chain_encoding_all: shape = [B, L] (integer chain encoding number per residue)
    #   features.all_chain_ids_per_entry: shape = list[list[str]] (ragged, all chain IDs per entry)
    #   features.visible_chain_ids_per_entry: shape = list[list[str]] (ragged, fixed context chains per entry)
    #   features.masked_chain_ids_per_entry: shape = list[list[str]] (ragged, designed chains per entry)
    #   features.masked_chain_lengths_per_entry: shape = list[list[int]] (ragged, lengths of designed chains per entry)
    #   features.chain_M_pos: shape = [B, L] (position-level design mask)
    #   features.omit_AA_mask: shape = [B, L, 21] (1.0 marks a forbidden amino acid at that position)
    #   features.residue_idx: shape = [B, L] (int64 per-residue numbering, -100 for padding)
    #   features.dihedral_mask: shape = [B, L, 3] (phi/psi/omega masks)
    #   features.tied_position_groups_per_entry: shape = list[list[list[int]]] (ragged tied-position groups per entry)
    #   features.pssm_coef_all: shape = [B, L]; features.pssm_bias_all: shape = [B, L, 21]; features.pssm_log_odds_all: shape = [B, L, 21]
    #   features.bias_by_res_all: shape = [B, L, 21]; features.tied_position_weight: shape = [B, L]
    features: TiedFeatures = tied_featurize(
        pdb_dict_list=batch_clones,
        device=device,
        chain_id_dict=chain_id_dict,
        fixed_position_dict=fixed_positions_dict,
        omit_AA_dict=omit_AA_dict,
        tied_positions_dict=tied_positions_dict,
        pssm_dict=pssm_dict,
        bias_by_res_dict=bias_by_res_dict,
    )
    # pssm_log_odds_mask: 0/1 mask of PSSM entries above the log-odds threshold,
    # shape = [B, L, 21] (1.0 for true, 0.0 for false)
    pssm_log_odds_mask: torch.Tensor = (features.pssm_log_odds_all > pssm_threshold).float() #1.0 for true, 0.0 for false
    # name_: name of the current structure (e.g. PDB code), shape = scalar str
    name_: str = batch_clones[0]['name']

    # randn_1: random decoding order seed for scoring the native sequence,
    # shape = [B, L] (same as chain_M)
    randn_1: torch.Tensor = torch.randn(features.chain_M.shape, device=features.X.device)
    # log_probs: per-position log probabilities of the native sequence
    # (chain mask = chain_M*chain_M_pos), shape = [B, L, 21]
    log_probs: torch.Tensor = model(
        X=features.X,
        S=features.S,
        mask=features.mask,
        chain_M=features.chain_M*features.chain_M_pos,
        residue_idx=features.residue_idx,
        chain_encoding_all=features.chain_encoding_all,
        randn=randn_1,
    )
    # mask_for_loss: combined mask so only designable positions contribute to the score,
    # shape = [B, L]
    mask_for_loss: torch.Tensor = features.mask * features.chain_M * features.chain_M_pos
    # scores: negative log probability of the native sequence at designable positions,
    # shape = [B] (one score per batch copy)
    scores: torch.Tensor = _scores(
        residue_type_indices=features.S,
        log_probs=log_probs,
        mask=mask_for_loss,
    )
    # native_score: native-sequence scores moved to CPU numpy, shape = [B]
    native_score: np.ndarray = scores.cpu().data.numpy()

    # Sample at every requested temperature (sampling_temperature: shape = scalar float)
    for sampling_temperature in temperatures:
        # Repeat sampling for every batch (j: shape = scalar batch index)
        for j in range(NUM_BATCHES):
            # randn_2: fresh random decoding order for this sampling pass,
            # shape = [B, L] (same as chain_M)
            randn_2: torch.Tensor = torch.randn(features.chain_M.shape, device=features.X.device)
            # Without tied positions, sample each position independently
            if tied_positions_dict == None:
                # Standard autoregressive sampling over all designable residues.
                # Arguments: X shape = [B, L, 4, 3]; randn_2 shape = [B, L];
                #   S shape = [B, L]; chain_M shape = [B, L];
                #   chain_encoding_all shape = [B, L]; residue_idx shape = [B, L]; mask shape = [B, L];
                #   chain_M_pos shape = [B, L]; omit_AA_mask shape = [B, L, 21]; pssm_coef shape = [B, L];
                #   pssm_bias / pssm_log_odds_mask shape = [B, L, 21]; bias_by_res shape = [B, L, 21]
                # Returns sample_dict: "S" shape = [B, L]; "probs" shape = [B, L, 21]; "decoding_order" shape = [B, L]
                sample_dict: dict[str, torch.Tensor] = model.sample(
                    X=features.X,
                    randn=randn_2,
                    S_true=features.S,
                    chain_mask=features.chain_M,
                    chain_encoding_all=features.chain_encoding_all,
                    residue_idx=features.residue_idx,
                    mask=features.mask,
                    temperature=sampling_temperature,
                    omit_AAs_np=omit_AAs_np,
                    bias_AAs_np=bias_AAs_np,
                    chain_M_pos=features.chain_M_pos,
                    omit_AA_mask=features.omit_AA_mask,
                    pssm_coef=features.pssm_coef_all,
                    pssm_bias=features.pssm_bias_all,
                    pssm_multi=pssm_multi,
                    pssm_log_odds_flag=bool(pssm_log_odds_flag),
                    pssm_log_odds_mask=pssm_log_odds_mask,
                    pssm_bias_flag=bool(pssm_bias_flag),
                    bias_by_res=features.bias_by_res_all,
                )
                # sampled_sequence: the sampled sequence as amino-acid indices, shape = [B, L]
                sampled_sequence: torch.Tensor = sample_dict["S"]
            else:
                # Tied sampling: tied positions share one probability computation and draw.
                # Same arguments as model.sample, plus tied_pos = tied_position_groups_per_entry[0]
                #   shape = list[list[int]] (tied position groups, 1-based chain-local indices)
                #   and tied_position_weight shape = [B, L]
                # Returns sample_dict: "S" shape = [B, L]; "probs" shape = [B, L, 21]; "decoding_order" shape = [B, L]
                sample_dict = model.tied_sample(
                    X=features.X,
                    randn=randn_2,
                    S_true=features.S,
                    chain_mask=features.chain_M,
                    chain_encoding_all=features.chain_encoding_all,
                    residue_idx=features.residue_idx,
                    mask=features.mask,
                    temperature=sampling_temperature,
                    omit_AAs_np=omit_AAs_np,
                    bias_AAs_np=bias_AAs_np,
                    chain_M_pos=features.chain_M_pos,
                    omit_AA_mask=features.omit_AA_mask,
                    pssm_coef=features.pssm_coef_all,
                    pssm_bias=features.pssm_bias_all,
                    pssm_multi=pssm_multi,
                    pssm_log_odds_flag=bool(pssm_log_odds_flag),
                    pssm_log_odds_mask=pssm_log_odds_mask,
                    pssm_bias_flag=bool(pssm_bias_flag),
                    tied_pos=features.tied_position_groups_per_entry[0],
                    tied_position_weight=features.tied_position_weight,
                    bias_by_res=features.bias_by_res_all,
                )
                # sampled_sequence: the sampled sequence (identical across tied chains), shape = [B, L]
                sampled_sequence = sample_dict["S"]
            # log_probs: per-position log probabilities of the sampled sequence
            # (same decoding_order as the sampling pass), shape = [B, L, 21]
            log_probs = model(
                X=features.X,
                S=sampled_sequence,
                mask=features.mask,
                chain_M=features.chain_M*features.chain_M_pos,
                residue_idx=features.residue_idx,
                chain_encoding_all=features.chain_encoding_all,
                randn=randn_2,
                use_input_decoding_order=True,
                decoding_order=sample_dict["decoding_order"],
            )
            # mask_for_loss: position mask for scoring the sampled sequence, shape = [B, L]
            mask_for_loss = features.mask*features.chain_M*features.chain_M_pos
            # scores: score of the sampled sequence (-log prob at designable positions), shape = [B]
            scores = _scores(
                residue_type_indices=sampled_sequence,
                log_probs=log_probs,
                mask=mask_for_loss,
            )
            # scores: sampled-sequence scores moved to CPU numpy, shape = [B]
            scores = scores.cpu().data.numpy()
            # all_probs_list entry: temperature-adjusted sampling probabilities, shape = [B, L, 21]
            all_probs_list.append(sample_dict["probs"].cpu().data.numpy())
            # all_log_probs_list entry: raw log probabilities, shape = [B, L, 21]
            all_log_probs_list.append(log_probs.cpu().data.numpy())
            # sampled_sequence_list entry: the sampled sequence, shape = [B, L]
            sampled_sequence_list.append(sampled_sequence.cpu().data.numpy())
            # Process every copy in the batch independently (b_ix: shape = scalar copy index)
            for b_ix in range(BATCH_COPIES):
                # masked_chain_lengths: lengths of each designed (masked) chain in this copy,
                # shape = list[int] (one scalar per designed chain)
                masked_chain_lengths: list[int] = features.masked_chain_lengths_per_entry[b_ix]
                # masked_chain_ids: chain IDs of the designed chains in internal order,
                # shape = list[str]
                masked_chain_ids: list[str] = features.masked_chain_ids_per_entry[b_ix]
                # position_match: 1.0 where the native and sampled residue types agree, 0.0 otherwise,
                # shape = [L] (elementwise equality of the two residue-index vectors)
                position_match: torch.Tensor = (features.S[b_ix] == sampled_sequence[b_ix]).to(torch.float32)
                # seq_recovery_rate: fraction of designable positions where the sample matches the native sequence,
                # shape = scalar tensor (position_match kept by mask_for_loss[b_ix] shape = [L], averaged over valid positions)
                seq_recovery_rate: torch.Tensor = torch.sum(position_match * mask_for_loss[b_ix]) / torch.sum(mask_for_loss[b_ix])
                # sample_seq: the sampled sequence decoded into one-letter codes, shape = scalar str
                sample_seq: str = _S_to_seq(sampled_sequence[b_ix], features.chain_M[b_ix])
                # score: score of this sampled sequence, shape = scalar float
                score: float = scores[b_ix]
                # Record the score for this sample
                score_list.append(score)
                # native_seq: the native sequence decoded for comparison, shape = scalar str
                native_seq: str = _S_to_seq(features.S[b_ix], features.chain_M[b_ix])
                # Chain boundaries in the concatenated layout, in masked_chain_ids order;
                # shape = [num_designed_chains + 1], e.g. [0, 320, 640] for two 320-residue chains
                chain_boundaries: np.ndarray = np.concatenate(([0], np.cumsum(masked_chain_lengths)))
                # For the very first sample only, also print the native sequence header
                if b_ix == 0 and j==0 and sampling_temperature==temperatures[0]:
                    # Per-chain native segments, in masked_chain_ids order (already sorted ascending)
                    native_segments: list[str] = [
                        native_seq[start_offset:end_offset]
                        for start_offset, end_offset in zip(chain_boundaries[:-1], chain_boundaries[1:])
                    ]
                    # Join the per-chain segments with '/' separators
                    native_seq = "/".join(native_segments)
                    # Chain IDs shown in the FASTA header (tied_featurize already sorts them)
                    print_masked_chains: list[str] = masked_chain_ids
                    print_visible_chains: list[str] = features.visible_chain_ids_per_entry[0]
                    # native_score_print: mean native-sequence score formatted to 4 significant digits,
                    # shape = scalar str
                    native_score_print: str = np.format_float_positional(np.float32(native_score.mean()), unique=False, precision=4)
                    # line: FASTA-style header with score/chains/model plus the native sequence,
                    # shape = scalar str
                    line: str = f'>{name_}, score={native_score_print}, fixed_chains={print_visible_chains}, designed_chains={print_masked_chains}, model_name={model_name}\n{native_seq}\n'
                    # Print header + native sequence without trailing blank line
                    print(line.rstrip())
                # Per-chain sampled segments, in masked_chain_ids order (already sorted ascending)
                sampled_segments: list[str] = [
                    sample_seq[start_offset:end_offset]
                    for start_offset, end_offset in zip(chain_boundaries[:-1], chain_boundaries[1:])
                ]
                # Join the per-chain segments with '/' separators
                sample_seq = "/".join(sampled_segments)
                # score_print: this sample's score formatted to 4 significant digits, shape = scalar str
                score_print: str = np.format_float_positional(np.float32(score), unique=False, precision=4)
                # seq_rec_print: the sequence-recovery rate formatted to 4 significant digits, shape = scalar str
                seq_rec_print: str = np.format_float_positional(np.float32(seq_recovery_rate.detach().cpu().numpy()), unique=False, precision=4)
                # line: FASTA-style header with temperature/sample/score/recovery plus the sampled sequence,
                # shape = scalar str
                line: str = f'>T={sampling_temperature}, sample={b_ix}, score={score_print}, seq_recovery={seq_rec_print}\n{sample_seq}\n'
                # Print header + sampled sequence
                print(line.rstrip())


# Collect the sampled data for visualization; the *_list containers are re-created inside
# the per-structure loop, so only the last structure's samples reach this concatenation
# all_probs_concat: probabilities concatenated across temperatures and batches,
# shape = [N, L, 21] where N = len(temperatures) * NUM_BATCHES * B
all_probs_concat: np.ndarray = np.concatenate(all_probs_list)
# all_log_probs_concat: raw log probabilities concatenated, shape = [N, L, 21] (same N as above)
all_log_probs_concat: np.ndarray = np.concatenate(all_log_probs_list)
# sampled_sequence_concat: sampled sequences concatenated, shape = [N, L] (same N as above)
sampled_sequence_concat: np.ndarray = np.concatenate(sampled_sequence_list)

# Print the sampling configuration and the shapes of the concatenated outputs;
# labels are left-aligned to a fixed width so the values line up
sampling_summary: list[tuple[str, object]] = [
    ("temperatures", temperatures),
    ("NUM_BATCHES", NUM_BATCHES),
    ("BATCH_COPIES", BATCH_COPIES),
    ("all_probs_concat", all_probs_concat.shape),
    ("all_log_probs_concat", all_log_probs_concat.shape),
    ("sampled_sequence_concat", sampled_sequence_concat.shape),
]
for label, value in sampling_summary:
    print(f"{label:<24} = {value}")


Generating sequences...
>1O91, score=1.2842, fixed_chains=[], designed_chains=['A', 'B', 'C'], model_name=v_48_020
EMPAFTAELTVPFPPVGAPVKFDKLLYNGRQNYNPQTGIFTCEVPGVYYFAYHVHCKGGNVWVALFKNNEPMMYTYDEYKKGFLDQASGSAVLLLRPGDQVFLQMPSEQAAGLYAGQYVHSSFSGYLLYPM/EMPAFTAELTVPFPPVGAPVKFDKLLYNGRQNYNPQTGIFTCEVPGVYYFAYHVHCKGGNVWVALFKNNEPMMYTYDEYKKGFLDQASGSAVLLLRPGDQVFLQMPSEQAAGLYAGQYVHSSFSGYLLYPM/EMPAFTAELTVPFPPVGAPVKFDKLLYNGRQNYNPQTGIFTCEVPGVYYFAYHVHCKGGNVWVALFKNNEPMMYTYDEYKKGFLDQASGSAVLLLRPGDQVFLQMPSEQAAGLYAGQYVHSSFSGYLLYPM
>T=0.1, sample=0, score=0.6603, seq_recovery=0.4656
TVEAFTALLTTPNPAVGTPIKFNSLVYNGGNVYDTATGIFTCKTPGIYLFQFVLNVYGNDLHVVLYKNDTPVLKVYLQNVDGKINQASGAAILELKKGDKVYVKIPSSSANGLYASSDNHSYFSGYLLTEL/TVEAFTALLTTPNPAVGTPIKFNSLVYNGGNVYDTATGIFTCKTPGIYLFQFVLNVYGNDLHVVLYKNDTPVLKVYLQNVDGKINQASGAAILELKKGDKVYVKIPSSSANGLYASSDNHSYFSGYLLTEL/TVEAFTALLTTPNPAVGTPIKFNSLVYNGGNVYDTATGIFTCKTPGIYLFQFVLNVYGNDLHVVLYKNDTPVLKVYLQNVDGKINQASGAAILELKKGDKVYVKIPSSSANGLYASSDNHSYFSGYLLTEL


In [ ]:
#@markdown ### Amino acid probabilties
# Plotly Express provides the heatmap drawing helper used in this cell
import plotly.express as px
# Average the per-sequence amino acid probabilities over the N sampled sequences (axis 0),
# then transpose so that rows = 21 amino acid types and columns = L sequence positions,
# shape = [21, L]
amino_acid_probability_matrix: np.ndarray = np.exp(all_log_probs_concat).mean(0).T
# Heatmap figure returned by Plotly Express;
# labels: dict[str, str] the axis titles, y: list[str] the 21 amino acid one-letter codes,
# template: str the Plotly theme name
fig = px.imshow(
    amino_acid_probability_matrix,
    labels=dict(x="positions", y="amino acids", color="probability"),
    y=list(alphabet),
    template="simple_white"
)

# Move the position axis to the top so the columns read in sequence order
fig.update_xaxes(side="top")

print(type(fig))

# Render the figure inline in the notebook output
fig.show()

In [7]:
#@markdown ### Sampling temperature adjusted amino acid probabilties
import plotly.express as px
fig = px.imshow(all_probs_concat.mean(0).T,
                labels=dict(x="positions", y="amino acids", color="probability"),
                y=list(alphabet),
                template="simple_white"
               )

fig.update_xaxes(side="top")


fig.show()